In [1]:
%pip install ipykernel torch transformer_lens einops

/Users/manavdahra/workspace/arena-course/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from dataclasses import dataclass
import torch
from torch import nn, Tensor
from jaxtyping import Float, Int
import einops
import math
from transformer_lens import HookedTransformer
from transformer_lens.utils import gelu_new, tokenize_and_concatenate
import datasets
from torch.utils.data import DataLoader
from transformers.models.gpt2.tokenization_gpt2_fast import GPT2TokenizerFast
import numpy as np
from tqdm import tqdm

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)

/Users/manavdahra/workspace/arena-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Transformer from scratch course

<img src="./images/transformer_arch.png" alt="image.png" style="width: 800px;"/>

### Tokenization and Embedding

- Input tokens ($t$) are integers (token IDs)
- Text is converted to tokens with a tokenizer (eg byte pair encoding)
- Token embedding is a lookup table that converts token IDs to vectors. Implemented by $W_E$ matrix. 
- The matrix consists of a stack of vectors one for each token in the vocabulary.

### Residual stream

Shape: `[batch, seq_len, d_model]`, where `d_model` is the length of a single embedding vector.

- The residual stream is the sum of all previous outputs of layers of the model, and is also the input to each new layer. 
- The initial value of the residual stream is denoted $x_0$ in the diagram, and $x_i$ are later values of the residual stream (after more attention and MLP layers have been applied to the residual stream).
- The residual stream is *really* fundamental. It's the central object of the transformer. It's how model remembers things, moves information between layers for composition, and it's the medium used to store the information that attention moves between positions.
- A key idea of transformers is the residual stream as output accumulation. As we move through the layers of the model, shifting information around and processing it, the values in the residual stream represent the accumulation of all the inferences made by the transformer up to that point.
- This is neatly illustrated by the logit lens. Rather than getting predictions from the residual stream at the very end of the model, we can take the value of the residual stream midway through the model and convert it to a distribution over tokens. When we do this, we find surprisingly coherent predictions, especially in the last few layers before the end

In [3]:
@dataclass
class Config:
    d_model: int = 768
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12

cfg = Config()
print(cfg)

Config(d_model=768, debug=True, layer_norm_eps=1e-05, d_vocab=50257, init_range=0.02, n_ctx=1024, d_head=64, d_mlp=3072, n_heads=12, n_layers=12)


## Test methods

- Generic test method to check shape of output from any later

In [4]:
from typing import Tuple


def test_layer_shape(cls, input: Tensor, expected_shape: Tuple):
    cfg = Config(debug=True)
    layer = cls(cfg)
    output = layer(input)
    assert isinstance(output, Tensor)
    assert expected_shape == output.shape, f"Output shape: {output.shape} doesnt match expected shape: {expected_shape}"

"""Test class to check if test_layer_shape works or not.
"""
class Identity(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

    def forward(self, x: Float[Tensor, "batch posn"]) -> Float[Tensor, "batch posn"]:
        return x

expected_shape = (5, 10)
test_layer_shape(Identity, torch.randn(expected_shape, device=device), expected_shape)

### Transformer blocks

<img src="./images/transformer_block.png" alt="image.png" style="width: 600px;"/>


### Layer norm
<img src="./images/layer_norm.png" width="600px">

- Make mean 0.
- Make variance 1. Ensure numerical stability with epsilon.
- Scale with learned weights. (multiply by gamma)
- Translate with learned bias. (add beta)

$$
\mu = \frac{1}{d_{model}} \sum_{i=1}^{d_{model}} x_i
$$

$$
\sigma = \sqrt{\frac{1}{d_{model}} \sum_{i=1}^{d_{model}} (x_i - \mu)^2 + \epsilon}
$$

$$
\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sigma} + \beta
$$

In [5]:
class LayerNorm(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.w = nn.Parameter(torch.ones(cfg.d_model, device=device))
        self.b = nn.Parameter(torch.zeros(cfg.d_model, device=device))
        
    def forward(self, res: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        mean = res.mean(dim=-1, keepdim=True)
        std = torch.sqrt(res.var(dim=-1, keepdim=True, unbiased=False) + self.cfg.layer_norm_eps)

        return ((res - mean) / std) * self.w + self.b

expected_shape = (1, 30, 768)
test_layer_shape(LayerNorm, torch.randn((1, 30, 768), device=device), expected_shape)

## Embed layer

This is a lookup table that converts token IDs to residual stream vectors. Implemented by $W_E$ matrix.

In [6]:
class Embed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_E = nn.Parameter(torch.empty((cfg.d_vocab, cfg.d_model), device=device))
        nn.init.normal_(self.W_E, std=self.cfg.init_range)
    
    def forward(self, tokens: Float[Tensor, "batch posn"]) -> Float[Tensor, "batch posn d_model"]:
        return self.W_E[tokens]

expected_shape = (1, 30, cfg.d_model)
test_layer_shape(Embed, torch.randint(0, cfg.d_vocab, (1, 30), device=device), expected_shape)

## Positional Embedding layer
- Positional embedding can also be thought of as a lookup table, but rather than the indices being our token IDs, the indices are just the numbers 0, 1, 2, ..., seq_len-1 (i.e. the position indices of the tokens in the sequence).

In [7]:
class PosEmbed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_pos = nn.Parameter(torch.empty((cfg.n_ctx, cfg.d_model), device=device))
        nn.init.normal_(self.W_pos, std=self.cfg.init_range)
    
    def forward(self, tokens: Int[Tensor, "batch posn"]) -> Float[Tensor, "batch posn d_model"]:
        """Lookup in W_pos (n_ctx x d_model) 
        to get an output tensor of shape (batch posn d_model)
        
        This implies we need to build index tensor (idx) of shape (batch x posn)
        doing self.W_pos[idx] yields output shape (batch x posn x d_model)
        """
        batch, posn = tokens.shape
        idx = torch.tile(torch.arange(0, posn), (batch, 1)).to(device)
        return self.W_pos[idx]

expected_shape = (2, 4, 768)
test_layer_shape(PosEmbed, torch.randn((2, 4), device=device), expected_shape)

### Applying causal mask

The causal mask function will be a method of the Attention class. It will take in attention scores, and apply a mask to them so that the model can only attend to previous positions (i.e. the model can't cheat by looking at future positions). We will implement this function first, and test it, before moving onto the forward method of the Attention class.

Common useful pytorch functions - 
1. [torch.where](https://www.google.com/url?q=https%3A%2F%2Fpytorch.org%2Fdocs%2Fstable%2Fgenerated%2Ftorch.where.html)
2. [torch.triu](https://www.google.com/url?q=https%3A%2F%2Fpytorch.org%2Fdocs%2Fstable%2Fgenerated%2Ftorch.triu.html)
3. [torch.masked_fill_](https://www.google.com/url?q=https%3A%2F%2Fpytorch.org%2Fdocs%2Fstable%2Fgenerated%2Ftorch.Tensor.masked_fill.html)

In [8]:
class Attention0(nn.Module):
    IGNORE: Float[Tensor, ""]
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.register_buffer("IGNORE", torch.tensor(float("-inf"), dtype=torch.float32, device=device))
    
    def apply_causal_mask(
            self,
            attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"]
    ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        query_pos, key_pos = attn_scores.shape[-2], attn_scores.shape[-1]

        mask = torch.ones((query_pos, key_pos), device=device)
        mask = torch.tril(mask).bool()

        return torch.where(
            mask,
            attn_scores,
            self.IGNORE,
        )

def test_attn_layer():
    test_input = torch.randn((2, 4, 8, 8), device=device)
    expected_output = test_input[0, 0, 0] # First row of (batch, posn, query) tensor
    expected_output[1:] = float("-inf")
    attn_layer = Attention0(cfg)
    test_output = attn_layer.apply_causal_mask(test_input)
    assert test_input.shape == test_output.shape, f"output shape: {test_output.shape} doesn't match {test_input.shape}"
    assert torch.allclose(expected_output, test_output[0, 0, 0]), f"output tensor: {test_output[0, 0, 0]} doesn't match {expected_output}"

test_attn_layer()


## Attention head

<img src="./images/attn_head_zoomed.png" width="800px">

In [9]:
class Attention(nn.Module):
    IGNORE: Float[Tensor, ""]

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head), device=device))
        self.W_K = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head), device=device))
        self.W_V = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head), device=device))
        self.W_O = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_head, cfg.d_model), device=device))

        self.b_Q = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head), device=device))
        self.b_K = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head), device=device))
        self.b_V = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head), device=device))
        self.b_O = nn.Parameter(torch.zeros((cfg.d_model), device=device))
        
        nn.init.normal_(self.W_Q, std=self.cfg.init_range)
        nn.init.normal_(self.W_K, std=self.cfg.init_range)
        nn.init.normal_(self.W_V, std=self.cfg.init_range)
        nn.init.normal_(self.W_O, std=self.cfg.init_range)
        self.register_buffer("IGNORE", torch.tensor(float("-inf"), dtype=torch.float32, device=device))

    
    def forward(self, res: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        """Project from embedding space to attention space
        """
        Q = einops.einsum(res, self.W_Q, "batch posn d_model,n_heads d_model d_head -> batch posn n_heads d_head") + self.b_Q
        K = einops.einsum(res, self.W_K, "batch posn d_model,n_heads d_model d_head -> batch posn n_heads d_head") + self.b_K
        V = einops.einsum(res, self.W_V, "batch posn d_model,n_heads d_model d_head -> batch posn n_heads d_head") + self.b_V

        """A = softmax(Q^T.K/sqrt(d_head))
        """
        attn_score = einops.einsum(Q, K, "batch query_pos n_heads d_head,batch key_pos n_heads d_head -> batch n_heads query_pos key_pos")
        attn_score = attn_score / math.sqrt(self.cfg.d_head) # normalize to avoid vanishing gradients
        attn_score = self.apply_causal_mask(attn_score)
        attn_score = attn_score.softmax(dim=-1) # softmax

        """Z = A*V
        """
        z = einops.einsum(attn_score, V, "batch n_heads query_pos key_pos,batch key_pos n_heads d_head -> batch n_heads query_pos d_head") 

        """Project from attention space to embedding space
        """
        return einops.einsum(z, self.W_O, "batch n_heads query_pos d_head,n_heads d_head d_model -> batch query_pos d_model") + self.b_O
    
    def apply_causal_mask(self, attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"]) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        query_pos, key_pos = attn_scores.shape[-2:]
        mask = torch.tril(torch.ones((query_pos, key_pos), device=device)).bool()
        return torch.where(
            mask,
            attn_scores,
            self.IGNORE,
        )

expected_shape = (2, 4, 768)
test_layer_shape(Attention, torch.randn(expected_shape, device=device), expected_shape)

## MLP layer

<img src="./images/mlp.png" width="600px">

In [10]:
class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_in = nn.Parameter(torch.empty((cfg.d_model, cfg.d_mlp), device=device))
        self.b_in = nn.Parameter(torch.zeros((cfg.d_mlp), device=device))
        self.W_out = nn.Parameter(torch.empty((cfg.d_mlp, cfg.d_model), device=device))
        self.b_out = nn.Parameter(torch.zeros((cfg.d_model), device=device))
        nn.init.normal_(self.W_in, std=self.cfg.init_range)
        nn.init.normal_(self.W_out, std=self.cfg.init_range)
    
    def forward(self, res: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        pre = einops.einsum(res, self.W_in, "batch posn d_model,d_model d_mlp -> batch posn d_mlp") + self.b_in
        mid = gelu_new(pre)
        post = einops.einsum(mid, self.W_out, "batch posn d_mlp,d_mlp d_model -> batch posn d_model") + self.b_out

        return post

expected_shape = (2, 4, 768)
test_layer_shape(MLP, torch.randn(expected_shape, device=device), expected_shape)

## Transformer block


In [11]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.ln1 = LayerNorm(cfg)
        self.attn = Attention(cfg)
        self.ln2 = LayerNorm(cfg)
        self.mlp = MLP(cfg)
    
    def forward(self, res: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        res_attn = res + self.attn(self.ln1(res))
        res_mlp = res_attn + self.mlp(self.ln2(res_attn))
        return res_mlp

expected_shape = (2, 4, cfg.d_model)
test_layer_shape(TransformerBlock, torch.randn(expected_shape, device=device), expected_shape)

## UnEmbed

In [12]:
class Unembed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_U = nn.Parameter(torch.empty((cfg.d_model, cfg.d_vocab), device=device))
        nn.init.normal_(self.W_U, std=self.cfg.init_range)
        self.b_U = nn.Parameter(torch.zeros((cfg.d_vocab), device=device, requires_grad=False))
    
    def forward(self, res: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_vocab"]:
        return einops.einsum(
            res, 
            self.W_U, 
            "batch posn d_model,d_model d_vocab -> batch posn d_vocab",
        ) + self.b_U

expected_shape = (2, 4, cfg.d_vocab)
test_layer_shape(Unembed, torch.randn((2, 4, 768), device=device), expected_shape)

## Complete demo transformer

In [13]:
class DemoTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = LayerNorm(cfg)
        self.unembed = Unembed(cfg)
    
    def forward(self, tokens: Int[Tensor, "batch posn"]) -> Float[Tensor, "batch posn d_vocab"]:
        res = self.embed(tokens) + self.pos_embed(tokens)
        for block in self.blocks:
            res = block(res)
        
        return self.unembed(self.ln_final(res))

expected_shape = (2, 4, cfg.d_vocab)
test_layer_shape(DemoTransformer, torch.randint(0, cfg.d_vocab, (2, 4), device=device), expected_shape)

### Testing demo transformer on trained weights

In [14]:
gpt2 = HookedTransformer.from_pretrained(
    "gpt2-small",
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,
    device=device,
)

demo_gpt2 = DemoTransformer(Config(debug=False))
demo_gpt2.load_state_dict(gpt2.state_dict(), strict=False)

text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = gpt2.to_tokens(text)
demo_logits = demo_gpt2(tokens)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer


In [15]:
def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    log_probs = logits.log_softmax(dim=-1)
    # Get logprobs the first seq_len-1 predictions (so we can compare them with the actual next tokens)
    log_probs_for_tokens = (
        log_probs[:, :-1].gather(dim=-1, index=tokens[:, 1:].unsqueeze(-1)).squeeze(-1)
    )

    return log_probs_for_tokens


pred_log_probs = get_log_probs(demo_logits, tokens)
print(f"Avg cross entropy loss: {-pred_log_probs.mean():.4f}")
print(f"Avg cross entropy loss for uniform distribution: {math.log(demo_gpt2.cfg.d_vocab):4f}")
print(f"Avg probability assigned to correct token: {pred_log_probs.exp().mean():4f}")

Avg cross entropy loss: 4.5647
Avg cross entropy loss for uniform distribution: 10.824905
Avg probability assigned to correct token: 0.087913


In [16]:
test_string = """Mitigating the risk of extinction from AI should be a global priority alongside other societal-scale risks such as"""
for i in range(100):
    test_tokens = gpt2.to_tokens(test_string)
    demo_logits = demo_gpt2(test_tokens)
    test_string += gpt2.tokenizer.decode(demo_logits[-1, -1].argmax())

print(test_string)

Mitigating the risk of extinction from AI should be a global priority alongside other societal-scale risks such as climate change, the spread of infectious diseases and the spread of infectious diseases.


The research is published in the journal Nature Communications.


The research team is led by Dr. Michael J. H. Haldane, a professor of biology at the University of California, Berkeley, and co-author of the paper.


"We are very excited to see that the AI community is starting to take notice of the potential for AI to be a major threat to the human race,"


## Training

Rough outline of training loop -
- Feed a batch of training data to the model (batch posn tokens)
- Collect the batch output of the model (batch posn logits)
- Calculate cross-entropy loss between true labels and model output
- `loss.backward()` - compute gradients of loss wrt to model parameters
- `optimizer.step()` - update model parameters using the computed gradients
- `optimizer.zero_grad()` - zero the gradients so they don't accumulate

We should encapsulate the training and validation procedures in a trainer class and use dataclass to setup trainer configuration.

In [17]:
model_cfg = Config(
    debug=False,
    d_model=32,
    n_heads=16,
    d_head=2,
    d_mlp=32 * 4,
    n_layers=4,
    n_ctx=128,
    d_vocab=gpt2.cfg.d_vocab,
)
model = DemoTransformer(model_cfg)

First we create a configuration file for controlling the training process

In [18]:
@dataclass
class TransformerTrainingArgs:
    batch_size: int = 32
    epochs: int = 10
    max_steps_per_epoch: int = 500
    lr: float = 1e-3
    weight_decay: float = 1e-2
    wandb_project: str | None = "day1-demotransformer"
    wandb_name: str | None = None


args = TransformerTrainingArgs()

We can now train the DemoTransformer using [Tinystories](https://huggingface.co/datasets/roneneldan/TinyStories) open source dataset. This dataset was created to research how small can a transformer model be.

In [19]:
dataset = datasets.load_dataset("roneneldan/TinyStories", split="train")
print(dataset)
print(dataset[0]["text"])

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Dataset({
    features: ['text'],
    num_rows: 2119719
})
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


Now let's create a tokenized dataset before feeding it into the model. We'll use `tokenize_and_concatenate` function to do this for us.

In [24]:
tokenized_dataset = tokenize_and_concatenate(
    dataset,
    gpt2.tokenizer,
    streaming=False,
    max_length=model_cfg.n_ctx,
    column_name="text",
    add_bos_token=True,
    num_proc=4,
)

dataset_dict = tokenized_dataset.train_test_split(test_size=1000)
train_loader = DataLoader(
    dataset_dict["train"], batch_size=args.batch_size, shuffle=True, num_workers=4, pin_memory=True
)
test_loader = DataLoader(
    dataset_dict["test"], batch_size=args.batch_size, shuffle=False, num_workers=4, pin_memory=True
)

Map (num_proc=4):   0%|          | 0/2119719 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (12536 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (10666 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (12536 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (10666 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (12297 > 1024). Running this sequence through the model will result in indexing errors
Token i

Map (num_proc=4):   0%|          | 0/2119719 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (12536 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (10666 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (12536 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (10666 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (12297 > 1024). Running this sequence through the model will result in indexing errors
Token i

TimeoutError: 

The transformed training data contains batch (size: 32) of dictionary items with single key - `tokens`

In [21]:
first_batch = train_loader.dataset[: args.batch_size]

print(first_batch.keys())
print(first_batch["tokens"].shape)

dict_keys(['tokens'])
torch.Size([32, 1024])


In [22]:
class TransformerSampler:
    def __init__(self, model: DemoTransformer, tokenizer: GPT2TokenizerFast):
        self.model = model
        self.cfg = model.cfg
        self.tokenizer = tokenizer

    @torch.inference_mode()
    def sample(self, prompt: str, max_tokens_generated=100, verbose=False, **kwargs) -> str:
        """
        Returns a string of autoregressively generated text, starting from the prompt.

        Sampling terminates at max_tokens_generated, or when the model generates an end-of-sequence token. kwargs are
        passed to sample_next_token, to give detailed instructions on how new tokens are chosen.
        """
        self.model.eval()
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)[0]

        for _ in range(max_tokens_generated):
            # Get new logits (make sure we don't pass in more tokens than the model's context length)
            logits = self.model(input_ids[None, -self.cfg.n_ctx :])
            # We only take logits for the last token, because this is what we're sampling
            logits = logits[0, -1]
            # Get next token (as a tensor of size (1, 1) so we can concat it to input_ids)
            next_token = torch.tensor(
                [TransformerSampler.sample_next_token(input_ids, logits, **kwargs)], device=device
            )
            # Create new input ids string, with shape (1, old_seq_len + 1)
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            # Print out results, if required
            if verbose:
                print(self.tokenizer.decode(input_ids), end="\r")
            # If our new token was the end-of-text token, stop
            if next_token == getattr(self.tokenizer, "eos_token_id", None):
                break

        return self.tokenizer.decode(input_ids)

    @staticmethod
    def sample_next_token(
        input_ids: Int[Tensor, "seq_len"],
        logits: Float[Tensor, "d_vocab"],
        temperature=1.0,
        top_k=0,
        top_p=0.0,
        frequency_penalty=0.0,
        seed=None,
    ) -> int:
        assert input_ids.ndim == 1, "input_ids should be a 1D sequence of token ids"
        assert temperature >= 0, "Temperature should be non-negative"
        assert 0 <= top_p <= 1.0, "Top-p must be a probability"
        assert 0 <= top_k, "Top-k must be non-negative"
        assert not (top_p != 0 and top_k != 0), "At most one of top-p and top-k supported"

        # Set random seeds for reproducibility
        if seed is not None:
            torch.manual_seed(seed)
            np.random.seed(seed)

        # Apply all the specialized sampling methods
        if temperature == 0:
            return TransformerSampler.greedy_search(logits)
        elif temperature != 1.0:
            logits = TransformerSampler.apply_temperature(logits, temperature)
        if frequency_penalty != 0.0:
            logits = TransformerSampler.apply_frequency_penalty(
                input_ids, logits, frequency_penalty
            )
        if top_k > 0:
            return TransformerSampler.sample_top_k(logits, top_k)
        if top_p > 0.0:
            return TransformerSampler.sample_top_p(logits, top_p)
        return TransformerSampler.sample_basic(logits)

    @staticmethod
    def greedy_search(logits: Float[Tensor, "d_vocab"]) -> int:
        """
        Returns the most likely token (as an int).
        """
        return logits.argmax().item()

    @staticmethod
    def apply_temperature(
        logits: Float[Tensor, "d_vocab"], temperature: float
    ) -> Float[Tensor, "d_vocab"]:
        """
        Applies temperature scaling to the logits.
        """
        return logits / temperature

    @staticmethod
    def apply_frequency_penalty(
        input_ids: Int[Tensor, "seq_len"], logits: Float[Tensor, "d_vocab"], freq_penalty: float
    ) -> Float[Tensor, "d_vocab"]:
        """
        Applies a frequency penalty to the logits.
        """
        d_vocab = logits.size(0)
        id_freqs = torch.bincount(input_ids, minlength=d_vocab)
        return logits - freq_penalty * id_freqs

    @staticmethod
    def sample_basic(logits: Float[Tensor, "d_vocab"]) -> int:
        """
        Samples from the distribution defined by the logits.
        """
        return torch.distributions.categorical.Categorical(logits=logits).sample().item()

    @staticmethod
    def sample_top_k(logits: Float[Tensor, "d_vocab"], k: int) -> int:
        """
        Samples from the top k most likely tokens.
        """
        top_k_logits, top_k_token_ids = logits.topk(k)
        # Get sampled token (which is an index corresponding to the list of top-k tokens)
        sampled_token_idx = torch.distributions.categorical.Categorical(logits=top_k_logits).sample()
        # Get the actual token id, as an int
        return top_k_token_ids[sampled_token_idx].item()

    @staticmethod
    def sample_top_p(
        logits: Float[Tensor, "d_vocab"], top_p: float, min_tokens_to_keep: int = 1
    ) -> int:
        """
        Samples from the most likely tokens which make up at least p cumulative probability.
        """
        # Sort logits, and get cumulative probabilities
        logits_sorted, indices = logits.sort(descending=True, stable=True)
        cumul_probs = logits_sorted.softmax(-1).cumsum(-1)
        # Choose which tokens to keep, in the set we sample from
        n_keep = torch.searchsorted(cumul_probs, top_p, side="left").item() + 1
        n_keep = max(n_keep, min_tokens_to_keep)
        keep_idx = indices[:n_keep]
        keep_logits = logits[keep_idx]
        # Perform the sampling
        sample = torch.distributions.categorical.Categorical(logits=keep_logits).sample()
        return keep_idx[sample].item()

    @torch.inference_mode()
    def beam_search(
        self,
        prompt: str,
        num_return_sequences: int,
        num_beams: int,
        max_new_tokens: int,
        no_repeat_ngram_size: int | None = None,
    ) -> list[tuple[float, str]]:
        """
        Implements a beam search, by repeatedly performing the `generate` and `filter` steps (starting from the initial
        prompt) until either of the two stopping criteria are met: (1) we've generated `max_new_tokens` tokens, or (2)
        we've generated `num_returns_sequences` terminating sequences.
        """
        raise NotImplementedError()

### Training loop


In [23]:
class TransformerTrainer:
    def __init__(self, args: TransformerTrainingArgs, model: DemoTransformer):
        super().__init__()
        self.model = model
        self.args = args
        self.sampler = TransformerSampler(self.model, gpt2.tokenizer)
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(), lr=args.lr, weight_decay=args.weight_decay
        )
        self.step = 0

        self.train_loader = DataLoader(
            dataset_dict["train"],
            batch_size=args.batch_size,
            shuffle=True,
            num_workers=4,
            pin_memory=True,
        )
        self.test_loader = DataLoader(
            dataset_dict["test"],
            batch_size=args.batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
        )

    def training_step(self, batch: dict[str, Int[Tensor, "batch seq"]]) -> Float[Tensor, ""]:
        """
        Calculates the loss on the tokens in the batch, performs a gradient update step, and logs the loss.

        Remember that `batch` is a dictionary with the single key 'tokens'.
        """
        tokens = batch["tokens"].to(device)
        outputs = self.model(tokens)
        loss = -get_log_probs(outputs, tokens).mean()
        loss.backward()

        self.step += 1
        self.optimizer.step()
        self.optimizer.zero_grad()
        
        return loss

    @torch.inference_mode()
    def evaluate(self) -> float:
        """
        Evaluate the model on the test set and return the accuracy.
        """
        correct, total = 0., 0
        self.model.eval()
        
        for i, batch in enumerate(self.test_loader):
            tokens = batch["tokens"].to(device)
            logits = self.model(tokens)[:,:-1]
            preds = logits.argmax(dim=-1)
            correct += (preds == tokens[:,1:]).sum().item()
            total += tokens.shape[0] * (tokens.shape[1] - 1)
        
        accuracy = correct/total
        print(f"Accuracy: {accuracy}")
        self.model.train()
        
        return accuracy

    def train(self):
        """
        Trains the model, for `self.args.epochs` epochs. Also handles wandb initialisation, and early stopping
        for each epoch at `self.args.max_steps_per_epoch` steps.
        """
        # wandb.init(project=self.args.wandb_project, name=self.args.wandb_name, config=self.args)
        accuracy = np.nan

        progress_bar = tqdm(total=self.args.max_steps_per_epoch * self.args.epochs)

        for epoch in range(self.args.epochs):
            for i, batch in enumerate(self.train_loader):
                loss = self.training_step(batch)
                progress_bar.update()
                progress_bar.set_description(
                    f"Epoch {epoch + 1}, loss: {loss:.3f}, accuracy: {accuracy:.3f}"
                )
                if i >= self.args.max_steps_per_epoch:
                    break

            accuracy = self.evaluate()
            sample_text = self.sampler.sample("Once upon a time", max_tokens_generated=50)
            print(sample_text)

        # wandb.finish()


# See the full run here: https://api.wandb.ai/links/dquarel/nrxuwnv7
model = DemoTransformer(model_cfg).to(device)
args = TransformerTrainingArgs()
trainer = TransformerTrainer(args, model)
trainer.train()

  0%|          | 0/5000 [00:00<?, ?it/s]/Users/manavdahra/workspace/arena-course/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


IndexError: index 128 is out of bounds for dimension 0 with size 128

## Evaluate model

TODO: Implement evaluation method to compute accuracy on test set.

### Save model

In [29]:
PATH = "../data/models/demo-gpt2.pt"

torch.save(model.state_dict(), PATH)

### Load model from state dict file

In [30]:
loaded_model = DemoTransformer(model_cfg)
loaded_model.load_state_dict(torch.load(PATH))
loaded_model.to(device)
loaded_model.eval()

DemoTransformer(
  (embed): Embed()
  (pos_embed): PosEmbed()
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (ln1): LayerNorm()
      (attn): Attention()
      (ln2): LayerNorm()
      (mlp): MLP()
    )
  )
  (ln_final): LayerNorm()
  (unembed): Unembed()
)